# Hazard Combination Exposure — Analysis

**Inputs**
- `HazardCombination_final.csv` — processed combination file (all countries, bitmask decoded)
- `data/misc/CCRI_P1_P2_format.geojson` — pipeline output with UNICEF region, income group, u18 population

**Sections**
1. Load final CSV and merge metadata
2. Top 20 combinations by exposed children
3. Stats by hazard level
4. Stats by UNICEF region
5. Stats by World Bank income group
6. Export all charts to PDF

In [ ]:
import pandas as pd
import geopandas as gpd
import json
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

# ── Economist palette ─────────────────────────────────────────────────────────
EC_BG    = '#D7E3EE'
EC_RED   = '#E3120B'
EC_BLACK = '#121212'
EC_GREY  = '#6B6B6B'
EC_GRID  = '#B8CBDB'
EC_WHITE = '#F5F5F5'

LEVEL_COLORS = {
    'Single':   '#A8C8E0',
    'Double':   '#3A7EB5',
    'Triple':   '#E07030',
    'Multiple': '#B82E2E',
}

def econ_style(ax, title='', subtitle='', xlabel='', ylabel=''):
    ax.set_facecolor(EC_BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis='both', length=0, labelsize=8.5, labelcolor=EC_BLACK)
    ax.yaxis.grid(True, color=EC_GRID, linewidth=0.8, linestyle='-', zorder=0)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=8, color=EC_GREY, labelpad=4)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=8, color=EC_GREY, labelpad=4)
    if title:
        ax.set_title(title, fontsize=11, fontweight='bold', color=EC_BLACK, loc='left', pad=8)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.01), xycoords='axes fraction', fontsize=8, color=EC_GREY, va='bottom')

def econ_fig(nrows=1, ncols=1, figsize=(10, 5), title='', subtitle=''):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    fig.patch.set_facecolor(EC_WHITE)
    fig.add_artist(plt.Line2D([0.01, 0.99], [0.99, 0.99],
                              transform=fig.transFigure, color=EC_RED, linewidth=2))
    if title:
        fig.text(0.01, 0.96, title, fontsize=12, fontweight='bold', color=EC_BLACK, va='top')
    if subtitle:
        fig.text(0.01, 0.91, subtitle, fontsize=8.5, color=EC_GREY, va='top')
    return fig, axes

def econ_legend(ax, loc='upper right'):
    handles = [mpatches.Patch(color=LEVEL_COLORS[l], label=l) for l in LEVEL_ORDER]
    leg = ax.legend(handles=handles, loc=loc, fontsize=8,
                    frameon=True, framealpha=1, facecolor=EC_BG, edgecolor=EC_GRID)
    for t in leg.get_texts():
        t.set_color(EC_BLACK)

plt.rcParams.update({'figure.dpi': 140, 'font.family': 'DejaVu Sans', 'font.size': 9})

FINAL_CSV    = '../../data/misc/HazardCombination_final.csv'
PIPELINE_GEO = '../../data/misc/CCRI_P1_P2_format.geojson'
PDF_PATH     = 'hazard_combination_report.pdf'

LEVEL_ORDER = ['Single', 'Double', 'Triple', 'Multiple']

REGION_LABELS = {
    'EAPRO': 'East Asia & Pacific',   'ECARO': 'Europe & Central Asia',
    'ESARO': 'Eastern & Southern Africa', 'LACRO': 'Latin America & Caribbean',
    'MENARO': 'Middle East & North Africa', 'ROSA': 'South Asia',
    'WCARO': 'West & Central Africa'
}
INCOME_LABELS = {'LI': 'Low Income', 'LMI': 'Lower-Middle', 'UMI': 'Upper-Middle', 'HI': 'High Income'}
INCOME_ORDER  = ['LI', 'LMI', 'UMI', 'HI']

figures = []   # collect figures for PDF export

## 1. Parse & aggregate combination chunks

In [ ]:
gdf  = gpd.read_file(PIPELINE_GEO)
meta = gdf[['iso3', 'unicef_ro', 'wb_income', 'u18_pop']].copy()
meta.columns = ['ISO3', 'unicef_ro', 'wb_income', 'u18_pop']
valid_iso3 = gdf[gdf['mhc_ge3_abs'].notna()]['iso3'].tolist()

df_all = pd.read_csv(FINAL_CSV)
print(f'Before filter: {df_all["ISO3"].nunique()} countries, {df_all["population"].sum():,.0f} children')

df = df_all[df_all['ISO3'].isin(valid_iso3)].merge(meta, on='ISO3', how='left')
print(f'After filter:  {df["ISO3"].nunique()} countries, {df["population"].sum():,.0f} children')

FILTERED_CSV = FINAL_CSV.replace('.csv', '_filtered.csv')
df.to_csv(FILTERED_CSV, index=False)
print(f'Saved → {FILTERED_CSV}')

total_children = df['population'].sum()
print(f'Unique combinations: {df["combination_id"].nunique()}')

## 2. Top 20 combinations

In [ ]:
top20 = (df.groupby(['combination', 'hazardLevel'])['population'].sum()
           .reset_index().sort_values('population', ascending=False).head(20))
top20['share'] = top20['population'] / total_children * 100

print(f'{"Combination":<55} {"Level":<10} {"Children":>15} {"Share":>7}')
print('-' * 92)
for _, r in top20.iterrows():
    print(f'{r["combination"]:<55} {r["hazardLevel"]:<10} {r["population"]:>15,.0f} {r["share"]:>6.1f}%')

fig, ax = econ_fig(figsize=(11, 8),
    title='Top 20 hazard combinations by exposed children',
    subtitle='Millions of children under 18, 198 countries')
ax.set_facecolor(EC_BG)
bars = ax.barh(range(len(top20)), top20['population'] / 1e6,
               color=[LEVEL_COLORS[l] for l in top20['hazardLevel']], height=0.6, zorder=3)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['combination'], fontsize=8.5, color=EC_BLACK)
ax.invert_yaxis()
ax.xaxis.grid(True, color=EC_GRID, linewidth=0.8, linestyle='-', zorder=0)
ax.yaxis.grid(False)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0, labelsize=8.5)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax.set_xlabel('Children exposed', fontsize=8, color=EC_GREY)
ax.set_xlim(0, top20['population'].max() / 1e6 * 1.38)
for bar, (_, row) in zip(bars, top20.iterrows()):
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{w:.1f}M ({row["share"]:.1f}%)', va='center', fontsize=7.5, color=EC_GREY)
econ_legend(ax, loc='lower right')
plt.tight_layout(rect=[0, 0.01, 1, 0.90])
figures.append(fig)
plt.show()

## 3. Stats by hazard level

In [ ]:
level_stats = (df.groupby('hazardLevel')['population'].sum()
                 .reindex(LEVEL_ORDER).reset_index())
level_stats['share'] = level_stats['population'] / total_children * 100

print(f'{"Level":<12} {"Children":>18} {"Share":>8}')
print('-' * 42)
for _, r in level_stats.iterrows():
    print(f'{r["hazardLevel"]:<12} {r["population"]:>18,.0f} {r["share"]:>7.1f}%')
print('-' * 42)
print(f'{"Total":<12} {total_children:>18,.0f} {100:>7.1f}%')

fig, (ax_bar, ax_pie) = econ_fig(1, 2, figsize=(11, 5),
    title='Children by hazard exposure level',
    subtitle=f'Total: {total_children/1e9:.2f} billion children under 18, 198 countries')

bars = ax_bar.bar(level_stats['hazardLevel'], level_stats['population'] / 1e6,
                  color=[LEVEL_COLORS[l] for l in level_stats['hazardLevel']], width=0.5, zorder=3)
for bar, (_, row) in zip(bars, level_stats.iterrows()):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 6,
                f'{row["population"]/1e6:.0f}M\n{row["share"]:.1f}%',
                ha='center', fontsize=8, color=EC_BLACK, linespacing=1.4)
econ_style(ax_bar, subtitle='Millions of children', ylabel='Millions')
ax_bar.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax_bar.set_ylim(0, level_stats['population'].max() / 1e6 * 1.3)
ax_bar.tick_params(axis='x', labelsize=9, labelcolor=EC_BLACK)

_, _, autotexts = ax_pie.pie(
    level_stats['population'], labels=None,
    colors=[LEVEL_COLORS[l] for l in level_stats['hazardLevel']],
    autopct='%1.1f%%', startangle=90, pctdistance=0.78,
    wedgeprops=dict(width=0.50, edgecolor=EC_WHITE, linewidth=1.5))
for at in autotexts:
    at.set_fontsize(8.5); at.set_color('white'); at.set_fontweight('bold')
ax_pie.text(0, 0, f'{total_children/1e9:.2f}B', ha='center', va='center',
            fontsize=11, color=EC_BLACK, fontweight='bold')
ax_pie.set_facecolor(EC_WHITE)
handles = [mpatches.Patch(color=LEVEL_COLORS[l], label=l) for l in LEVEL_ORDER]
ax_pie.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.1),
              ncol=2, fontsize=8, frameon=True, facecolor=EC_WHITE, edgecolor=EC_GRID)

plt.tight_layout(rect=[0, 0.01, 1, 0.88])
figures.append(fig)
plt.show()

## 4. Stats by UNICEF region

In [ ]:
region_level = (df.groupby(['unicef_ro', 'hazardLevel'])['population'].sum()
                  .unstack(fill_value=0).reindex(columns=LEVEL_ORDER, fill_value=0))
region_level['Total'] = region_level.sum(axis=1)
region_level = region_level.sort_values('Total', ascending=False)

print(f'{"Region":<28}', '  '.join(f'{l:>12}' for l in LEVEL_ORDER), f'{"Total":>14}')
print('-' * 90)
for ro, row in region_level.iterrows():
    label = REGION_LABELS.get(ro, ro)
    vals  = '  '.join(f'{row[l]/1e6:>11.1f}M' for l in LEVEL_ORDER)
    print(f'{label:<28} {vals}  {row["Total"]/1e6:>12.1f}M')

fig, (ax_abs, ax_pct) = econ_fig(1, 2, figsize=(14, 5.5),
    title='Children by hazard level and UNICEF region',
    subtitle='Millions of children under 18 (left); share within each region (right)')

short_labels = [REGION_LABELS.get(r, r).replace(' & ', '\n& ') for r in region_level.index]

for ax, use_pct, ylabel in [(ax_abs, False, 'Millions'), (ax_pct, True, 'Share (%)')]:
    denom  = region_level['Total'] if use_pct else pd.Series(1e6, index=region_level.index)
    bottom = np.zeros(len(region_level))
    for level in LEVEL_ORDER:
        vals = (region_level[level] / denom).values
        bars = ax.bar(short_labels, vals, bottom=bottom, label=level,
                      color=LEVEL_COLORS[level], width=0.55, zorder=3,
                      edgecolor=EC_WHITE, linewidth=0.4)
        for bar, v in zip(bars, vals):
            if v > (7 if use_pct else 8):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_y() + bar.get_height() / 2,
                        f'{v:.0f}{"%%" if use_pct else "M"}',
                        ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')
        bottom += vals
    econ_style(ax, ylabel=ylabel)
    fmt = (lambda x, _: f'{x:.0f}%') if use_pct else (lambda x, _: f'{x:.0f}M')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
    if use_pct:
        ax.set_ylim(0, 106)
    ax.tick_params(axis='x', labelsize=7.5)

econ_legend(ax_pct, loc='upper right')
plt.tight_layout(rect=[0, 0.01, 1, 0.88])
figures.append(fig)
plt.show()

## 5. Stats by World Bank income group

In [ ]:
income_level = (df.groupby(['wb_income', 'hazardLevel'])['population'].sum()
                  .unstack(fill_value=0)
                  .reindex(index=INCOME_ORDER, columns=LEVEL_ORDER, fill_value=0))
income_level['Total'] = income_level.sum(axis=1)
income_level['Triple+_share'] = (
    (income_level['Triple'] + income_level['Multiple']) / income_level['Total'] * 100)

print(f'{"Income Group":<22}', '  '.join(f'{l:>12}' for l in LEVEL_ORDER),
      f'{"Total":>14}  {"Triple+%":>9}')
print('-' * 100)
for inc, row in income_level.iterrows():
    vals = '  '.join(f'{row[l]/1e6:>11.1f}M' for l in LEVEL_ORDER)
    print(f'{INCOME_LABELS.get(inc,inc):<22} {vals}  {row["Total"]/1e6:>12.1f}M  {row["Triple+_share"]:>8.1f}%')

income_pct = income_level[LEVEL_ORDER].div(income_level['Total'], axis=0) * 100
xlabels = [INCOME_LABELS[i] for i in income_level.index]

fig = plt.figure(figsize=(13, 5.5))
fig.patch.set_facecolor(EC_WHITE)
fig.add_artist(plt.Line2D([0.01, 0.99], [0.99, 0.99],
               transform=fig.transFigure, color=EC_RED, linewidth=2))
fig.text(0.01, 0.96, 'Hazard combination exposure by World Bank income group',
         fontsize=12, fontweight='bold', color=EC_BLACK, va='top')
fig.text(0.01, 0.91, 'Millions of children (left); share within group (centre); Triple+ share (right)',
         fontsize=8.5, color=EC_GREY, va='top')

gs = GridSpec(1, 3, figure=fig, width_ratios=[2, 2, 1.3], wspace=0.38)
ax_abs = fig.add_subplot(gs[0])
ax_pct = fig.add_subplot(gs[1])
ax_dot = fig.add_subplot(gs[2])

for ax, use_pct, ylabel in [(ax_abs, False, 'Millions'), (ax_pct, True, 'Share (%)')]:
    denom  = income_level['Total'] if use_pct else pd.Series(1e6, index=income_level.index)
    bottom = np.zeros(4)
    for level in LEVEL_ORDER:
        vals = (income_level[level] / denom).values
        bars = ax.bar(xlabels, vals, bottom=bottom, label=level,
                      color=LEVEL_COLORS[level], width=0.5,
                      zorder=3, edgecolor=EC_WHITE, linewidth=0.4)
        for bar, v in zip(bars, vals):
            if v > (7 if use_pct else 15):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_y() + bar.get_height() / 2,
                        f'{v:.0f}{"%%" if use_pct else "M"}',
                        ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        bottom += vals
    econ_style(ax, ylabel=ylabel)
    fmt = (lambda x, _: f'{x:.0f}%') if use_pct else (lambda x, _: f'{x:.0f}M')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
    if use_pct:
        ax.set_ylim(0, 106)
    plt.setp(ax.get_xticklabels(), rotation=15, ha='right', fontsize=8.5)
econ_legend(ax_pct, loc='upper right')

ax_dot.set_facecolor(EC_BG)
for spine in ax_dot.spines.values():
    spine.set_visible(False)
ax_dot.tick_params(length=0)
triple_vals = income_level['Triple+_share'].values
y_pos = list(range(len(income_level) - 1, -1, -1))
ax_dot.axvline(50, color=EC_GRID, linewidth=1.2, zorder=1)
ax_dot.scatter(triple_vals, y_pos, color=EC_RED, s=80, zorder=3, clip_on=False)
for v, y in zip(triple_vals, y_pos):
    ax_dot.text(v + 2, y, f'{v:.1f}%', va='center', fontsize=9, color=EC_BLACK, fontweight='bold')
ax_dot.set_yticks(y_pos)
ax_dot.set_yticklabels([INCOME_LABELS[i] for i in income_level.index], fontsize=8.5, color=EC_BLACK)
ax_dot.set_xlim(0, 100)
ax_dot.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax_dot.xaxis.grid(True, color=EC_GRID, linewidth=0.8, zorder=0)
ax_dot.yaxis.grid(False)
ax_dot.set_axisbelow(True)
ax_dot.tick_params(axis='x', labelsize=8, labelcolor=EC_GREY)
ax_dot.set_title('Triple+ share', fontsize=10, fontweight='bold', color=EC_BLACK, loc='left', pad=8)
ax_dot.text(50, len(income_level) - 0.2, '50%', ha='center', fontsize=7.5, color=EC_GREY, fontstyle='italic')

plt.tight_layout(rect=[0, 0.01, 1, 0.88])
figures.append(fig)
plt.show()

## 6. Export to PDF

In [ ]:
with PdfPages(PDF_PATH) as pdf:
    for fig in figures:
        pdf.savefig(fig, bbox_inches='tight')
print(f'Saved {len(figures)} pages → {PDF_PATH}')